# TP 6 — Modèles linéaires pour la régression

**Cours :** *Linear Models for Regression* (slides 137–157) — modèle linéaire et **moindres carrés ordinaires** (OLS), théorème de **Gauss-Markov**, $\mathrm{TSS} = \mathrm{ESS} + \mathrm{RSS}$, $R^2$ ; cas dégénérés et **régularisation** (ridge, lasso) ; régression non linéaire **paramétrique** (fonctions de base) et **non paramétrique** (noyaux, théorème du représentant).

**À la fin de ce notebook, vous saurez :**
1. coder l'OLS (équations normales et QR), ses diagnostics ($R^2$, erreurs standard) et vérifier Gauss-Markov par simulation ;
2. estimer un **modèle factoriel** (Fama-French) — la régression la plus utilisée en finance quantitative ;
3. coder **ridge** (forme close + validation croisée LOO gratuite) et **lasso** (descente par coordonnées), et savoir lequel choisir quand on combine **beaucoup de signaux faibles** ;
4. faire de la régression non linéaire avec des **fonctions de base** et avec la **kernel ridge regression**.

> 📝 **Version énoncé.** Complétez les blocs `# TODO`, puis exécutez la cellule *Test* qui suit chaque exercice. Le corrigé est dans `../corriges/`.

In [ ]:
import sys, pathlib, warnings
_here = pathlib.Path.cwd()
ROOT = _here.parent if _here.name in ("exercices", "corriges") else _here
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tp_utils import *          # style graphique, check(), check_close(), timer()...
import tp_data                  # données réelles (cache / téléchargement) ou simulées

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
rng = np.random.default_rng(42)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.kernel_ridge import KernelRidge

## Partie 1 — Moindres carrés ordinaires

Modèle : $Y = [1\ X^\top]\beta + \varepsilon$, $\varepsilon$ centré indépendant de $X$. On minimise $\mathrm{RSS}(\beta) = \|y - X\beta\|^2$ (où $X$ contient une colonne de 1). Gradient $-2X^\top(y - X\beta)$, Hessienne $2X^\top X \succeq 0$ : si $X$ est de rang plein,
$$\hat\beta = (X^\top X)^{-1}X^\top y.$$

**En pratique**, on ne forme jamais $(X^\top X)^{-1}$ : on résout le système linéaire (`np.linalg.solve`), ou mieux on utilise une **décomposition QR** $X = QR$, d'où $R\hat\beta = Q^\top y$ — plus stable car le conditionnement de $X^\top X$ est le **carré** de celui de $X$.

### Exercice 1.1 — `ols_normal` et `ols_qr`

In [ ]:
def add_intercept(X):
    X = np.asarray(X, float).reshape(len(X), -1)
    return np.c_[np.ones(len(X)), X]


def ols_normal(X, y):
    """X inclut déjà la colonne de 1."""
    # TODO : np.linalg.solve(XᵀX, Xᵀy)
    raise NotImplementedError("À compléter")


def ols_qr(X, y):
    # TODO : np.linalg.qr (mode réduit), puis système triangulaire
    raise NotImplementedError("À compléter")

In [ ]:
# Tests
X = rng.normal(size=(300, 4)); y = 1.5 + X @ [2, -1, 0, 0.5] + rng.normal(0, 0.3, 300)
Xi = add_intercept(X)
sk = LinearRegression().fit(X, y)
check_close(ols_normal(Xi, y), np.r_[sk.intercept_, sk.coef_], "équations normales = sklearn")
check_close(ols_qr(Xi, y), np.r_[sk.intercept_, sk.coef_], "QR = sklearn")

### Expérience 1.2 — Pourquoi QR : un design presque colinéaire

Deux variables quasi identiques ($x_2 = x_1 + 10^{-7}\,\eta$). On compare l'erreur sur $\hat\beta$ quand les données sont **sans bruit** (la solution exacte est connue).

In [ ]:
x1 = rng.normal(size=500); x2 = x1 + 1e-7 * rng.normal(size=500)
Xc = add_intercept(np.c_[x1, x2]); beta_true = np.array([1.0, 2.0, 3.0]); yc = Xc @ beta_true
print(f"conditionnement de X : {np.linalg.cond(Xc):.1e} | de XᵀX : {np.linalg.cond(Xc.T @ Xc):.1e}")
for name, f in [("équations normales", ols_normal), ("QR", ols_qr)]:
    try:
        print(f"{name:20s} erreur max sur β : {np.abs(f(Xc, yc) - beta_true).max():.2e}")
    except np.linalg.LinAlgError as e:
        print(f"{name:20s} échec : {e}")

### Exercice 1.3 — Évaluation : $\mathrm{TSS} = \mathrm{ESS} + \mathrm{RSS}$, $R^2$, erreurs standard

- $\mathrm{TSS} = \sum(y_i - \bar y)^2$, $\mathrm{ESS} = \sum(\hat y_i - \bar y)^2$, $\mathrm{RSS} = \sum(y_i - \hat y_i)^2$ ; pour l'OLS avec constante, $\mathrm{TSS} = \mathrm{ESS} + \mathrm{RSS}$ (Pythagore : le résidu est orthogonal aux colonnes de $X$).
- $R^2 = 1 - \mathrm{RSS}/\mathrm{TSS} = 1 - \mathrm{FVU}$ (*fraction of variance unexplained*).
- Sous homoscédasticité : $\widehat{\mathrm{Var}}(\hat\beta) = \hat\sigma^2(X^\top X)^{-1}$ avec $\hat\sigma^2 = \mathrm{RSS}/(n - p)$, $p$ = nombre de colonnes de $X$ ; statistique $t_j = \hat\beta_j / \mathrm{se}_j$.

In [ ]:
def ols_report(X, y, names=None):
    """X inclut la constante. Retourne (DataFrame coef/se/t, dict TSS/ESS/RSS/R2)."""
    n, p = X.shape
    beta = ols_qr(X, y); yhat = X @ beta
    # TODO : TSS, ESS, RSS ; σ̂² = RSS / (n − p) ; se = sqrt(diag(σ̂² (XᵀX)⁻¹))
    raise NotImplementedError("À compléter")
    names = names or [f"b{j}" for j in range(p)]
    tab = pd.DataFrame({"coef": beta, "se": se, "t": beta / se}, index=names)
    return tab, {"TSS": TSS, "ESS": ESS, "RSS": RSS, "R2": 1 - RSS / TSS}

tab, dec = ols_report(Xi, y, ["const", "x1", "x2", "x3", "x4"])
check(np.isclose(dec["TSS"], dec["ESS"] + dec["RSS"]), "TSS = ESS + RSS")
check(np.isclose(dec["R2"], sk.score(X, y)), "R² = sklearn")
tab

### Expérience 1.4 — Gauss-Markov

**Théorème** : sous le modèle linéaire à bruit centré, homoscédastique et non corrélé, l'OLS est le meilleur estimateur **linéaire sans biais** (BLUE) : tout autre estimateur linéaire sans biais a une variance au moins aussi grande.

Vérifions-le : $X$ fixé, on tire 3000 bruits, et on compare l'OLS à un autre estimateur linéaire sans biais, les **moindres carrés pondérés** avec des poids arbitraires $\tilde\beta = (X^\top WX)^{-1}X^\top Wy$.

In [ ]:
Xf = add_intercept(rng.normal(size=(60, 2))); bt = np.array([0.5, 1.0, -2.0])
W = np.diag(rng.uniform(0.1, 3, 60))
A_ols = np.linalg.solve(Xf.T @ Xf, Xf.T); A_wls = np.linalg.solve(Xf.T @ W @ Xf, Xf.T @ W)
E = rng.normal(0, 1, (3000, 60)); Ys = Xf @ bt + E
B_ols, B_wls = Ys @ A_ols.T, Ys @ A_wls.T
print("biais moyen  OLS :", np.round(B_ols.mean(0) - bt, 3), "| WLS :", np.round(B_wls.mean(0) - bt, 3))
print("variance     OLS :", np.round(B_ols.var(0), 4), "| WLS :", np.round(B_wls.var(0), 4))
check(np.all(B_ols.var(0) <= B_wls.var(0) * 1.05), "Gauss-Markov : variance OLS ≤ variance d'un autre estimateur sans biais")

## Partie 2 — Application : le modèle factoriel de Fama-French

En finance, l'OLS sert d'abord à **décomposer** un rendement en expositions à des facteurs :
$$r_{i,t} - r_{f,t} = \alpha_i + \beta_{i,\mathrm{MKT}}\,(\mathrm{MKT}_t - r_{f,t}) + \beta_{i,\mathrm{SMB}}\,\mathrm{SMB}_t + \beta_{i,\mathrm{HML}}\,\mathrm{HML}_t + \varepsilon_{i,t}.$$
$\beta$ = expositions (risque systématique : marché, taille, value), $\alpha$ = rendement « anormal » non expliqué, $R^2$ = part du risque qui est factorielle.

**Données :** facteurs quotidiens de la Kenneth French Data Library et actions de `tp_data`. Si l'une des sources est simulée, on fabrique des actions synthétiques **à bêtas connus** à partir des facteurs : on peut alors vérifier que l'OLS les retrouve.

In [ ]:
ff, src_ff = tp_data.load_fama_french("3", start="2015-01-01")
rets, sectors, src_st = tp_data.load_stock_returns()
names = ["AAPL", "JPM", "XOM", "NEE", "WMT"]
if "simulé" in (src_ff, src_st):
    print("→ actions synthétiques à bêtas connus (une des sources est simulée)")
    true_b = pd.DataFrame({"alpha": [0.0] * 5, "Mkt-RF": [1.2, 1.1, 0.9, 0.5, 0.6],
                           "SMB": [-0.2, 0.1, 0.0, -0.1, -0.3], "HML": [-0.4, 0.9, 0.8, 0.2, 0.0]}, index=names)
    F = ff[["Mkt-RF", "SMB", "HML"]].to_numpy()
    excess = pd.DataFrame(F @ true_b[["Mkt-RF", "SMB", "HML"]].T.to_numpy()
                          + rng.normal(0, 0.012, (len(ff), 5)), index=ff.index, columns=names)
else:
    true_b = None
    common = rets.index.intersection(ff.index)
    excess = rets.loc[common, names].sub(ff.loc[common, "RF"], axis=0)
    ff = ff.loc[common]
print("sources :", src_ff, "/", src_st, "| jours :", len(excess))

In [ ]:
rows, tabs = [], {}
Xff = add_intercept(ff[["Mkt-RF", "SMB", "HML"]].to_numpy())
for tk in names:
    # TODO : ols_report, puis une ligne par action (alpha annualisé, t(alpha), bêtas, R²)
    raise NotImplementedError("À compléter")
ff_tab = pd.DataFrame(rows).set_index("ticker")
if true_b is not None:
    z = [(tabs[tk].loc[f, "coef"] - true_b.loc[tk, f]) / tabs[tk].loc[f, "se"] for tk in names for f in ["Mkt-RF", "SMB", "HML"]]
    check(np.max(np.abs(z)) < 4, f"l'OLS retrouve les bêtas simulés (écart max = {np.max(np.abs(z)):.1f} erreurs standard)")
ff_tab

**Lecture.** Sur données réelles : JPM et XOM ont un $\beta_{\mathrm{HML}}$ positif (valeurs « value »), AAPL négatif (« growth ») ; NEE (utility) a un bêta marché faible. Les $|t(\alpha)| < 2$ sont la norme : un alpha statistiquement significatif est rare.

**Attention aux erreurs standard** : les rendements sont hétéroscédastiques (clustering de volatilité) ; en pratique on utilise des erreurs standard **robustes** (White, Newey-West) — l'estimateur $\hat\beta$ reste le même, seule sa variance estimée change.

## Partie 3 — Cas dégénérés et régularisation

Si $X$ n'est pas de rang plein (variables colinéaires, ou $n < d$), il existe un sous-espace affine de minimiseurs. Deux remèdes : réduire la dimension (TP 10), ou **régulariser**.

- **Ridge** : $\min_\beta\|y - X\beta\|^2 + \lambda\|\beta\|_2^2$ → $\hat\beta = (X^\top X + \lambda D)^{-1}X^\top y$, strictement convexe pour $\lambda>0$. Convention pratique (scikit-learn) : on ne pénalise **pas** la constante, d'où $D = \mathrm{diag}(0, 1, \dots, 1)$.
- **Lasso** : $\min_\beta\frac1{2n}\|y - X\beta\|^2 + \lambda\|\beta\|_1$ — pas de forme close, mais produit des coefficients **exactement nuls** (sélection de variables).

### Exercice 3.1 — `ridge` et sa validation croisée LOO « gratuite »

Pour ridge, la matrice chapeau $H = X(X^\top X + \lambda D)^{-1}X^\top$ donne exactement les résidus leave-one-out : $y_i - \hat y_{-i} = \dfrac{y_i - \hat y_i}{1 - H_{ii}}$. Un seul ajustement suffit pour toute la LOO.

In [ ]:
def ridge(X, y, lam):
    """X inclut la constante (non pénalisée)."""
    # TODO : D = identité avec D[0, 0] = 0 ; résoudre (XᵀX + λD) β = Xᵀy
    raise NotImplementedError("À compléter")


def ridge_loo_mse(X, y, lam):
    # TODO : A = (XᵀX + λD)⁻¹ Xᵀ ; h_i = (X A)_ii (np.einsum) ; résidus LOO = (y − Xβ) / (1 − h)
    raise NotImplementedError("À compléter")

In [ ]:
# Tests
X = rng.normal(size=(80, 10)); y = X[:, 0] - X[:, 1] + rng.normal(0, 1, 80); Xi = add_intercept(X)
skr = Ridge(alpha=5.0).fit(X, y)
check_close(ridge(Xi, y, 5.0), np.r_[skr.intercept_, skr.coef_], "ridge = sklearn (constante non pénalisée)")
loo_brut = np.mean([(y[i] - add_intercept(X[i:i+1]) @ ridge(np.delete(Xi, i, 0), np.delete(y, i), 5.0)) ** 2
                    for i in range(len(y))])
check(np.isclose(ridge_loo_mse(Xi, y, 5.0), loo_brut), "formule LOO = LOO par force brute")

### Exercice 3.2 — `lasso_cd` : descente par coordonnées

On centre $X$ et $y$ (la constante se déduit à la fin). Pour chaque coordonnée $j$, avec le résidu partiel $r^{(j)} = y - X\beta + X_j\beta_j$ :
$$\beta_j \leftarrow \frac{S\big(\tfrac1n X_j^\top r^{(j)},\ \lambda\big)}{\tfrac1n\|X_j\|^2},\qquad S(z, \lambda) = \mathrm{sign}(z)\max(|z| - \lambda, 0)\ \ \text{(seuillage doux)}.$$
On répète les balayages jusqu'à ce que le plus grand changement soit inférieur à `tol`. Maintenez le résidu à jour plutôt que de le recalculer.

In [ ]:
def soft_threshold(z, lam):
    return np.sign(z) * np.maximum(np.abs(z) - lam, 0.0)


def lasso_cd(X, y, lam, max_iter=5000, tol=1e-8, beta0=None):
    """X sans constante. Retourne (intercept, beta)."""
    X = np.asarray(X, float); y = np.asarray(y, float)
    n, p = X.shape
    xm, ym = X.mean(0), y.mean()
    Xc, yc = X - xm, y - ym
    col_sq = (Xc ** 2).sum(0) / n
    beta = np.zeros(p) if beta0 is None else beta0.copy()
    resid = yc - Xc @ beta
    for it in range(max_iter):
        # TODO : pour chaque j : ρ = X_jᵀ r / n + col_sq[j]·β_j ; β_j ← S(ρ, λ) / col_sq[j] ; mettre à jour r ; arrêt si changement max < tol
        raise NotImplementedError("À compléter")
    return ym - xm @ beta, beta

In [ ]:
# Test
X = rng.normal(size=(200, 15)); y = 3 * X[:, 0] - 2 * X[:, 3] + rng.normal(0, 1, 200)
b0, b = lasso_cd(X, y, 0.1)
skl = Lasso(alpha=0.1, tol=1e-10, max_iter=100000).fit(X, y)
check_close(b, skl.coef_, "lasso_cd = sklearn", atol=1e-5)
check(np.sum(b != 0) < 15, f"le lasso met des coefficients exactement à 0 ({np.sum(b == 0)} nuls)")

### Expérience 3.3 — Chemins de régularisation

In [ ]:
X = rng.normal(size=(150, 8)) @ np.linalg.cholesky(0.6 * np.ones((8, 8)) + 0.4 * np.eye(8)).T
y = X @ [3, -2, 1.5, 0, 0, 0, 0, 0] + rng.normal(0, 2, 150)
lams_r = np.geomspace(1e-2, 1e4, 60); lams_l = np.geomspace(1e-3, 3, 60)
path_r = np.array([ridge(add_intercept(X), y, l)[1:] for l in lams_r])
path_l, warm = [], None
for l in lams_l[::-1]:
    _, warm = lasso_cd(X, y, l, beta0=warm); path_l.append(warm.copy())
path_l = np.array(path_l[::-1])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for j in range(8):
    style = dict(color=PALETTE[j], lw=1.6 if j < 3 else 1, ls="-" if j < 3 else ":")
    axes[0].semilogx(lams_r, path_r[:, j], **style, label=f"β{j+1}" if j < 4 else None)
    axes[1].semilogx(lams_l, path_l[:, j], **style)
axes[0].set_title("Ridge : rétrécissement continu vers 0"); axes[0].set_xlabel("λ")
axes[1].set_title("Lasso : coefficients annulés un à un"); axes[1].set_xlabel("λ")
axes[0].legend(title="pointillés : vrais β = 0", fontsize=8); plt.tight_layout(); plt.show()

### Application 3.4 — Combiner beaucoup de signaux faibles

Situation typique d'une équipe de recherche quantitative : $p = 60$ « signaux » (prédicteurs) **corrélés** entre eux, un rendement futur à prédire, et un **rapport signal/bruit très faible** ($R^2$ théorique de l'ordre de 2 %). On compare OLS, ridge et lasso **hors échantillon** dans deux mondes :
- **dense** : tous les signaux portent un peu d'information ;
- **sparse** : 5 signaux seulement sont utiles.

Mesure : $R^2$ hors échantillon $= 1 - \sum(y - \hat y)^2 / \sum(y - \bar y_{\text{train}})^2$ (négatif = pire que prédire la moyenne).

In [ ]:
def make_signals(n, p=60, n_fac=4, seed=0):
    r_ = np.random.default_rng(seed)
    L = r_.normal(0, 1, (n_fac, p))
    return r_.normal(size=(n, n_fac)) @ L * 0.7 + r_.normal(size=(n, p))

def scenario(kind, n_train=1500, n_test=5000, p=60, r2_target=0.02, seed=0):
    r_ = np.random.default_rng(seed)
    S = make_signals(n_train + n_test, p, seed=seed)
    beta = r_.normal(size=p) if kind == "dense" else np.r_[r_.normal(0, 1, 5) * 3, np.zeros(p - 5)]
    sig = S @ beta
    noise_sd = np.sqrt(sig.var() * (1 - r2_target) / r2_target)
    y = sig + r_.normal(0, noise_sd, len(sig))
    return S[:n_train], y[:n_train], S[n_train:], y[n_train:]

def oos_r2(y_te, pred, y_tr):
    return 1 - ((y_te - pred) ** 2).sum() / ((y_te - y_tr.mean()) ** 2).sum()

In [ ]:
def lasso_cv_lambda(X, y, lams, n_folds=5, seed=0):
    """λ minimisant l'erreur de CV ; chemin de λ décroissants avec démarrage à chaud (warm start)."""
    lams = np.sort(lams)[::-1]
    folds = np.array_split(np.random.default_rng(seed).permutation(len(y)), n_folds)
    cv = np.zeros(len(lams))
    for f in folds:
        tr = np.setdiff1d(np.arange(len(y)), f)
        warm = None
        for i, l in enumerate(lams):
            b0, warm = lasso_cd(X[tr], y[tr], l, tol=1e-6, beta0=warm)
            cv[i] += ((y[f] - b0 - X[f] @ warm) ** 2).sum()
    return lams[int(np.argmin(cv))]

rows = []
for kind in ["dense", "sparse"]:
    for seed in range(6):
        Xa, ya, Xb, yb = scenario(kind, seed=seed)
        Xai, Xbi = add_intercept(Xa), add_intercept(Xb)
        # TODO : OLS ; ridge avec λ choisi par LOO ; lasso avec λ choisi par lasso_cv_lambda
        raise NotImplementedError("À compléter")
        rows.append((kind, seed, r_ols, r_ridge, r_lasso))
res = pd.DataFrame(rows, columns=["monde", "seed", "OLS", "ridge", "lasso"])
summary = res.groupby("monde")[["OLS", "ridge", "lasso"]].mean()
check((summary["ridge"] > summary["OLS"]).all(), "la régularisation bat l'OLS hors échantillon dans les deux mondes")
print("R² hors échantillon moyen (en %) — vrai R² = 2 %")
(100 * summary).round(2)

**Questions.**
1. Pourquoi l'OLS a-t-il un $R^2$ hors échantillon négatif ou quasi nul alors que le vrai $R^2$ est de 2 % ?
2. Dans quel monde le lasso fait-il mieux que ridge, et inversement ? Pourquoi ?

> ✍️ *Votre réponse :* …

## Partie 4 — Régression non linéaire paramétrique : fonctions de base

On reste dans le cadre linéaire **en $\beta$** : $f_\beta(x) = \sum_j\beta_j\,\phi_j(x)$, avec des fonctions de base $\phi_j$ fixées (polynômes, splines…). L'objectif reste quadratique en $\beta$ : on réutilise OLS / ridge sur la matrice $\Phi_{ij} = \phi_j(x_i)$.

### Exercice 4.1 — Polynômes vs splines linéaires (fonctions « charnières »)
- `poly_features(x, deg)` : $[x, x^2, \dots, x^{\deg}]$ après mise à l'échelle de $x$ dans $[-1, 1]$ ;
- `hinge_features(x, knots)` : $[x, (x - t_1)_+, \dots, (x - t_K)_+]$ — fonction affine par morceaux continue (c'est exactement ce que calcule une couche de neurones ReLU, TP 9).

In [ ]:
def poly_features(x, deg, lo=-3, hi=3):
    # TODO : u = x ramené dans [−1, 1], puis u[:, None] ** np.arange(1, deg + 1)
    raise NotImplementedError("À compléter")


def hinge_features(x, knots):
    # TODO : colonnes x et max(x − t_k, 0) pour chaque nœud t_k
    raise NotImplementedError("À compléter")

f_true = lambda x: np.sin(2 * x) + 0.3 * x
x = rng.uniform(-3, 3, 60); y = f_true(x) + rng.normal(0, 0.3, 60)
xt = rng.uniform(-3, 3, 5000); yt = f_true(xt) + rng.normal(0, 0.3, 5000)
degs = list(range(1, 21)); n_knots = list(range(1, 21))
err_poly = [np.mean((yt - add_intercept(poly_features(xt, d)) @ ols_qr(add_intercept(poly_features(x, d)), y)) ** 2) for d in degs]
err_hinge = []
for K in n_knots:
    kn = np.linspace(-3, 3, K + 2)[1:-1]
    b = ols_qr(add_intercept(hinge_features(x, kn)), y)
    err_hinge.append(np.mean((yt - add_intercept(hinge_features(xt, kn)) @ b) ** 2))
check(min(err_poly) < 0.2 and min(err_hinge) < 0.2, "les deux bases peuvent approcher f")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(degs, err_poly, marker="o", label="polynômes (degré)")
axes[0].semilogy(n_knots, err_hinge, marker="o", label="splines linéaires (nb de nœuds)")
axes[0].axhline(0.09, color=GRAY, ls="--", lw=1); axes[0].text(1, 0.095, "bruit", color=INK2, fontsize=9)
axes[0].set_xlabel("complexité"); axes[0].set_title("Erreur de test (n = 60)"); axes[0].legend()
xs = np.linspace(-3, 3, 400)
axes[1].scatter(x, y, s=10, color=GRAY)
for i, d in enumerate([3, 15]):
    axes[1].plot(xs, add_intercept(poly_features(xs, d)) @ ols_qr(add_intercept(poly_features(x, d)), y),
                 color=PALETTE[i], label=f"polynôme de degré {d}")
axes[1].set_ylim(-3, 3); axes[1].legend(); axes[1].set_title("Sur-apprentissage des polynômes de haut degré")
plt.tight_layout(); plt.show()

## Partie 5 — Régression non paramétrique à noyaux (kernel ridge)

On plonge implicitement les données dans un espace de fonctions (RKHS) de noyau $k$, par exemple gaussien $k(x, x') = \exp(-\|x-x'\|^2/2\sigma^2)$. Le **théorème du représentant** dit que le minimiseur de $\sum_i(y_i - f(x_i))^2 + \lambda\|f\|_{\mathcal H}^2$ s'écrit $f^*(\cdot) = \sum_j\alpha_j\,k(x_j, \cdot)$. En notant $K_{ij} = k(x_i, x_j)$ :
$$\hat\alpha = (K + \lambda I_n)^{-1}y,\qquad \hat f(x) = \sum_j\hat\alpha_j\,k(x_j, x).$$
C'est l'**astuce du noyau** : on ne manipule jamais le plongement, seulement les produits scalaires $k(x_i, x_j)$. Coût : $O(n^3)$ en apprentissage, $O(n)$ par prédiction.

### Exercice 5.1

In [ ]:
def gaussian_kernel(A, B, sigma):
    A = np.asarray(A, float).reshape(len(A), -1); B = np.asarray(B, float).reshape(len(B), -1)
    # TODO : distances² par développement du carré, puis exp(−D² / 2σ²)
    raise NotImplementedError("À compléter")


def kernel_ridge_fit(X, y, sigma, lam):
    # TODO : α = (K + λ I)⁻¹ y
    raise NotImplementedError("À compléter")


def kernel_ridge_predict(Xtr, alpha, Xte, sigma):
    # TODO : k(X_test, X_train) @ α
    raise NotImplementedError("À compléter")

In [ ]:
# Test
X = rng.uniform(-2, 2, (100, 2)); y = np.sin(X[:, 0]) * X[:, 1] + rng.normal(0, 0.1, 100); Xq = rng.uniform(-2, 2, (20, 2))
a = kernel_ridge_fit(X, y, 0.7, 0.1)
ref = KernelRidge(alpha=0.1, kernel="rbf", gamma=1 / (2 * 0.7 ** 2)).fit(X, y).predict(Xq)
check_close(kernel_ridge_predict(X, a, Xq, 0.7), ref, "kernel ridge = sklearn")

### Expérience 5.2 — L'expérience du cours : influence de σ

$y = \mathrm{sinc}\big((x - 49.5)/3\big) + \varepsilon$, $x\in[0, 100]$, $\varepsilon\sim\mathcal N(0, 0.1^2)$, $\lambda$ fixé.

In [ ]:
xk = np.sort(rng.uniform(0, 100, 150)); u = (xk - 49.5) / 3
yk = np.sinc(u / np.pi) + rng.normal(0, 0.1, len(xk))
xs = np.linspace(0, 100, 600)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
for ax, s in zip(axes, [0.1, 1, 2.5, 10]):
    a = kernel_ridge_fit(xk, yk, s, 0.1)
    ax.scatter(xk, yk, s=6, color=GRAY)
    ax.plot(xs, kernel_ridge_predict(xk, a, xs, s), color=BLUE)
    ax.set_title(f"σ = {s}")
axes[0].set_ylim(-0.6, 1.4); plt.tight_layout(); plt.show()

**Lecture :** σ = 0,1 → surapprentissage (variance) ; σ = 10 → presque linéaire (biais) ; σ et λ se choisissent par validation croisée, comme $k$ pour le k-NN.

## Partie 6 — Synthèse : quel modèle de régression, quand ?

| Méthode | Paramétrique ? | Coût d'apprentissage | Hyperparamètres | Forces | Faiblesses | Quand l'utiliser |
|---|---|---|---|---|---|---|
| **OLS** | oui | $O(nd^2)$ (QR) | — | sans biais, BLUE (Gauss-Markov), interprétable, inférence ($t$-stats) | variance forte si $d$ grand / bruit fort ; dégénère si colinéarité | modèles factoriels, attribution, $n \gg d$ et bon rapport signal/bruit |
| **Ridge** | oui | $O(nd^2 + d^3)$, LOO gratuite | λ | stable, gère la colinéarité, excellent à faible SNR avec effets **denses** | biaisé, pas de sélection | combinaison de nombreux signaux corrélés (alpha combining), prévision |
| **Lasso** | oui | itératif (descente par coordonnées) | λ | **sélection** de variables, modèles parcimonieux | instable avec variables corrélées, biais sur les grands coefficients | quelques prédicteurs utiles parmi beaucoup ; interprétabilité |
| **Fonctions de base** | oui | comme OLS/ridge sur $\Phi$ | base, taille | non-linéarité tout en restant linéaire en β | choix de la base ; polynômes instables aux bords | non-linéarités de forme connue, 1–2 variables |
| **Kernel ridge** | non | $O(n^3)$ | noyau, σ, λ | très flexible, solution exacte en forme close | coût en $n$, malédiction de la dimension | $n$ ≲ 10⁴, relations non linéaires lisses |